# 018: Decorators — Closure Bindings, Stack-Based Execution, and Production Registry Engines

## 1. THEORY LAYER

### First-Class Functions and Closures
Python treats functions as **first-class citizens** — they can be passed as arguments, returned from other functions, and bound to variables.

A **closure** is a function that retains access to variables from its enclosing lexical scope even after the outer function has finished executing.

### Decorators Origin
A decorator is a structural design pattern that dynamically extends or alters the behavior of a function or method without modifying its source code.

---

## 2. CPYTHON INTERNAL LAYER

### Cell Objects and `__closure__`
When a nested function references an outer local variable, CPython turns that variable into a **cell object**.
- The outer function's frame stores the cell object.
- The inner function holds a reference to this cell in its `__closure__` tuple.
- This keeps the variable alive on the heap even after the outer function's execution frame is popped off the call stack.

---


In [ ]:
import sys
import time
from functools import wraps

print("=" * 70)
print("SECTION 1: Closure Memory Mechanics")
print("=" * 70)

def outer(msg):
    # msg is captured inside a cell object
    def inner():
        print(f"Message: {msg}")
    return inner

fn = outer("Hello World")
# Inspect closure cell objects
print(f"Closure cells: {fn.__closure__}")
print(f"Cell contents: {fn.__closure__[0].cell_contents}")



---
## 3. COMPLETE METHOD REFERENCE

Below is the implementation of various decorator patterns:
- **Stateful class-based decorators**: Preserves states between invocations.
- **Decorators accepting parameters**: Uses a three-level function nesting pattern.
- **Decorator chaining**: Execution flow order (outer-to-inner).


In [ ]:
# Stateful Class-Based Decorator
class CallCounter:
    """Decorator that counts function invocations using internal state."""
    def __init__(self, func):
        self.func = func
        self.count = 0
        wraps(func)(self)  # Preserve metadata

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCounter
def process_data(data):
    return f"Processed: {data}"

print("\n=== Stateful Class Decorator ===")
print(process_data("A"))
print(process_data("B"))
print(f"Total invocations: {process_data.count}")



### Decorators with Parameters
To accept parameters, a decorator must return a decorator, which in turn returns the wrapped function.


In [ ]:
def repeat(times):
    """Decorator factory repeating function execution 'times' times."""
    def decorator_repeat(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator_repeat

@repeat(times=3)
def greet(name):
    print(f"Hello, {name}")

print("\n=== Decorator with Parameters ===")
greet("Alice")



---
## 4. IMPLEMENTATION LAYER

### Level 3: Advanced Usage Systems — API Route Registry & Dispatcher
Below, we implement a production-grade Web API routing registry using decorators.


In [ ]:
class APIRouter:
    """Decoupled API Router matching URLs to handler functions via decorators."""
    def __init__(self):
        self.routes = {}

    def route(self, path, methods=["GET"]):
        """Decorator factory mapping a path to an API handler."""
        def decorator(func):
            for method in methods:
                self.routes[(path, method)] = func
            return func
        return decorator

    def dispatch(self, path, method="GET"):
        """Dispatches an incoming request to the matching handler."""
        key = (path, method)
        if key not in self.routes:
            return "404 Not Found"
        return self.routes[key]()

router = APIRouter()

@router.route("/users", methods=["GET"])
def get_users():
    return "[User1, User2]"

@router.route("/users", methods=["POST"])
def create_user():
    return "User Created"

print("\n=== Level 3: API Router Dispatcher ===")
print(f"GET /users:  {router.dispatch('/users', 'GET')}")
print(f"POST /users: {router.dispatch('/users', 'POST')}")
print(f"GET /admins: {router.dispatch('/admins', 'GET')}")



---
## 5. EXPERIMENTATION LAYER
### Decorator Chaining Execution Flow
We trace the execution order of multiple stacked decorators to show that evaluation flows from top (outer) to bottom (inner).


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Decorator Chaining Flow")
print("=" * 70)

def decorator_one(func):
    def wrapper(*args, **kwargs):
        print(" -> Entering Decorator One")
        res = func(*args, **kwargs)
        print(" <- Exiting Decorator One")
        return res
    return wrapper

def decorator_two(func):
    def wrapper(*args, **kwargs):
        print("   -> Entering Decorator Two")
        res = func(*args, **kwargs)
        print("   <- Exiting Decorator Two")
        return res
    return wrapper

@decorator_one
@decorator_two
def compute():
    print("     Executing Core Function")

compute()



---
## 6. VISUALIZATION LAYER
```
Stack-Based Decorator Chaining Call Flow:
[ Call compute() ]
       |
       v
[ wrapper_one (decorator_one) ]
       |
       v
[ wrapper_two (decorator_two) ]
       |
       v
[ Core compute() Function ]
```
---
## 7. INTERVIEW MASTER LAYER

### 100 Scenario-Based Decorator Interview Q&As (Summary Selection)

1. **Why is `functools.wraps` essential when writing decorators?**
   - Because a decorator returns a new wrapper function, which discards the original function's metadata (docstrings, name, signatures). `wraps` copies this metadata back to the wrapper.
2. **What is the difference between class-based decorators and function-based decorators?**
   - Function-based decorators use closures to maintain state. Class-based decorators use `__init__` and implement `__call__` to behave like callables.
3. **Explain the evaluation order of stacked decorators.**
   - They are evaluated bottom-up during compilation, and executed top-down (outer-to-inner) during runtime.
4. **How do you write a decorator that accepts parameters?**
   - By using a three-level nested function structure (a decorator factory), where the outer function accepts the decorator parameters, the middle accepts the function, and the inner accepts arguments.
5. **Can decorators be used on classes?**
   - Yes. A class decorator accepts a class object, modifies its attributes or overrides its instantiation rules, and returns the modified class.
